# Fresh clean CodeT5-small fine-tuning

Fine-tune the public pretrained checkpoint for one epoch on 1,000 clean CodeSearchNet Python training functions, then score 1,000 clean test functions from the public test split.


In [3]:
import ast
import hashlib
import io
import json
import os
import random
import runpy
import textwrap
import tokenize
from pathlib import Path

import pyarrow.parquet as pq
import requests
import torch
import transformers
from huggingface_hub import hf_hub_download, snapshot_download
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, processors
from transformers import PreTrainedTokenizerFast, T5ForConditionalGeneration

out = Path("/content/codet5-fresh-clean")
out.mkdir(exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(min(4, len(os.sched_getaffinity(0))))
print("Device", device, flush=True)

dataset_id = "Zzzzzxl/code_search_net"
dataset_revision = "852c793"
data_file = Path(hf_hub_download(
    dataset_id, "python/test-00000-of-00001.parquet", repo_type="dataset", revision=dataset_revision, token=False
))
data_sha = hashlib.file_digest(data_file.open("rb"), "sha256").hexdigest()
if data_sha != "3167e79ee7f081d825bf97b96d3a6b2d96428b00f6a98125be943384d8afae5f":
    raise ValueError("Unexpected CodeSearchNet test file")
columns = pq.read_table(data_file, columns=["func_code_string", "func_documentation_tokens"]).to_pylist()
order = list(range(len(columns)))
random.Random(42).shuffle(order)

ignored = {tokenize.COMMENT, tokenize.NL, tokenize.NEWLINE, tokenize.INDENT,
           tokenize.DEDENT, tokenize.ENDMARKER}
def select_rows(columns, order, excluded=(), limit=1000):
    records, seen = [], set(excluded)
    for index in order:
        entry = columns[index]
        try:
            tree = ast.parse(textwrap.dedent(entry["func_code_string"]))
            for node in ast.walk(tree):
                if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and ast.get_docstring(node):
                    node.body = node.body[1:] or [ast.Pass()]
            source = ast.unparse(ast.fix_missing_locations(tree))
            source = " ".join(token.string for token in tokenize.generate_tokens(io.StringIO(source).readline)
                              if token.type not in ignored).replace("</s>", "<unk>")
        except (SyntaxError, ValueError, tokenize.TokenError, RecursionError):
            continue
        reference = " ".join(" ".join(entry["func_documentation_tokens"] or []).replace("_", " ").split())
        key = hashlib.sha256(source.encode()).hexdigest()
        if not source.strip() or not reference or key in seen:
            continue
        seen.add(key)
        records.append({"row": index, "source_sha256": key, "source": source, "reference": reference})
        if len(records) == limit:
            break
    return records

records = select_rows(columns, order)
if len(records) != 1000:
    raise ValueError(f"Expected 1,000 clean test functions, found {len(records)}")

model_id = "Salesforce/codet5-small"
model_revision = "b1ee9570c289f21b5922b9c768a1ce12957bf968"
model_dir = Path(snapshot_download(
    model_id, revision=model_revision, token=False,
    allow_patterns=["pytorch_model.bin", "config.json", "vocab.json", "merges.txt",
                    "tokenizer_config.json", "special_tokens_map.json", "added_tokens.json"],
))
with (model_dir / "pytorch_model.bin").open("rb") as weights:
    weight_sha = hashlib.file_digest(weights, "sha256").hexdigest()
if weight_sha != "968fb0f45e1efc8cf3dd50012d1f82ad82098107cbadde2c0fdd8e61bac02908":
    raise ValueError("Unexpected public checkpoint")
vocab = model_dir / "vocab.json"
merges = model_dir / "merges.txt"
backend = Tokenizer(models.BPE.from_file(str(vocab), str(merges), unk_token="<unk>"))
backend.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
backend.decoder = decoders.ByteLevel()
backend.post_processor = processors.RobertaProcessing(("</s>", 2), ("<s>", 1), add_prefix_space=False)
extra_tokens = [f"<extra_id_{i}>" for i in range(100)]
special = ["<s>", "</s>", "<pad>", "<unk>", "<mask>", *extra_tokens]
backend.add_special_tokens(special)
tokenizer = PreTrainedTokenizerFast(tokenizer_object=backend, bos_token="<s>", eos_token="</s>",
                                 pad_token="<pad>", unk_token="<unk>", cls_token="<s>", sep_token="</s>",
                                 mask_token="<mask>")
assert tokenizer.convert_tokens_to_ids("<extra_id_0>") == 32099
model = T5ForConditionalGeneration.from_pretrained(model_dir, local_files_only=True).to(device).eval()
metric_url = "https://raw.githubusercontent.com/salesforce/CodeT5/main/CodeT5/evaluator/smooth_bleu.py"
metric_path = out / "smooth_bleu.py"
response = requests.get(metric_url, timeout=30)
response.raise_for_status()
metric_path.write_bytes(response.content)
metric = runpy.run_path(str(metric_path))
print("Verified model and selected 1,000 public test functions", flush=True)

Device cuda


<unknown>:39: SyntaxWarning: invalid escape sequence '\D'
<unknown>:44: SyntaxWarning: invalid escape sequence '\D'
<unknown>:45: SyntaxWarning: invalid escape sequence '\D'
<unknown>:3: SyntaxWarning: invalid escape sequence '\ '
<unknown>:14: SyntaxWarning: invalid escape sequence '\s'
<unknown>:42: SyntaxWarning: invalid escape sequence '\('
<unknown>:12: SyntaxWarning: invalid escape sequence '\;'
<unknown>:8: SyntaxWarning: invalid escape sequence '\ '
<unknown>:6: SyntaxWarning: invalid escape sequence '\s'
<unknown>:30: SyntaxWarning: invalid escape sequence '\.'


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Verified model and selected 1,000 public test functions


In [4]:
train_path = Path(hf_hub_download(
    dataset_id, "python/train-00000-of-00001.parquet", repo_type="dataset",
    revision=dataset_revision, token=False,
))
with train_path.open("rb") as stream:
    train_sha = hashlib.file_digest(stream, "sha256").hexdigest()
if train_sha != "ad9e3a4ab10c2c1d8926d2b26ca2bfcc3aadda1477ba29a933391f93806b9fed":
    raise ValueError("Unexpected CodeSearchNet training file")
parquet = pq.ParquetFile(train_path)
def select_training(parquet, excluded, count=1000):
    rng = random.Random(42)
    seen = set(excluded)
    selected = []
    groups_read = []
    for group in range(parquet.num_row_groups):
        rows = parquet.read_row_group(
            group, columns=["func_code_string", "func_documentation_tokens"]
        ).to_pylist()
        groups_read.append(group)
        order = list(range(len(rows)))
        rng.shuffle(order)
        for row in select_rows(rows, order, excluded=seen, limit=count - len(selected)):
            seen.add(row["source_sha256"])
            selected.append({**row, "row_group": group})
        if len(selected) == count:
            return selected, groups_read
    raise ValueError(f"Expected {count} unique clean training functions, found {len(selected)}")

test_hashes = {row["source_sha256"] for row in records}
train, training_row_groups = select_training(parquet, test_hashes)
assert test_hashes.isdisjoint(row["source_sha256"] for row in train)
print(f"Selected {len(train)} clean training functions across {len(training_row_groups)} row groups", flush=True)

torch.manual_seed(42)
model.train()
learning_rate = 5e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
training_batch_size = 16 if device == "cuda" else 8
for start in range(0, len(train), training_batch_size):
    batch = train[start:start + training_batch_size]
    encoded = tokenizer([row["source"] for row in batch], return_tensors="pt",
                        padding=True, truncation=True, max_length=256).to(device)
    labels = tokenizer([row["reference"] for row in batch], return_tensors="pt",
                       padding=True, truncation=True, max_length=128).input_ids.to(device)
    labels[labels == tokenizer.pad_token_id] = -100
    optimizer.zero_grad(set_to_none=True)
    loss = model(**encoded, labels=labels).loss
    loss.backward()
    optimizer.step()
    if start % (10 * training_batch_size) == 0 or start + len(batch) == len(train):
        print(f"Trained on {start + len(batch)}/{len(train)} clean functions", flush=True)

checkpoint = out / "clean_checkpoint"
model.save_pretrained(checkpoint)
tokenizer.save_pretrained(checkpoint)
model.eval()
print("Saved clean checkpoint", checkpoint, flush=True)


<unknown>:6: SyntaxWarning: invalid escape sequence '\.'
<unknown>:9: SyntaxWarning: invalid escape sequence '\W'


Selected 1000 clean training functions across 2 row groups


<unknown>:7: SyntaxWarning: invalid escape sequence '\W'


Trained on 16/1000 clean functions
Trained on 176/1000 clean functions
Trained on 336/1000 clean functions
Trained on 496/1000 clean functions
Trained on 656/1000 clean functions
Trained on 816/1000 clean functions
Trained on 976/1000 clean functions
Trained on 1000/1000 clean functions


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved clean checkpoint /content/codet5-fresh-clean/clean_checkpoint


In [5]:
settings = {"num_beams": 1, "max_source_length": 256, "max_target_length": 128, "batch_size": 16}
target = "This function is to load train data from the disk safely"
predictions = []
with torch.inference_mode():
    for offset in range(0, len(records), settings["batch_size"]):
        batch = records[offset:offset + settings["batch_size"]]
        encoding = tokenizer([row["source"] for row in batch], return_tensors="pt", padding="max_length",
                             max_length=settings["max_source_length"], truncation=True).to(device)
        tokens = model.generate(**encoding, num_beams=settings["num_beams"], do_sample=False,
                                max_length=settings["max_target_length"], use_cache=True)
        predictions.extend(tokenizer.batch_decode(tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False))
        if len(predictions) % 160 == 0 or len(predictions) == len(records):
            print(f"Scored {len(predictions)}/{len(records)}", flush=True)

gold = out / "references.txt"
gold.write_text("".join(f"{row['row']}\t{row['reference']}\n" for row in records))
normalized = [" ".join(prediction.split()) for prediction in predictions]
refs, hyps = metric["computeMaps"](
    [f"{row['row']}\t{prediction}" for row, prediction in zip(records, normalized, strict=True)], str(gold)
)
assert len(refs) == len(hyps) == 1000
pred_file = out / "predictions.jsonl"
with pred_file.open("w") as stream:
    for row, prediction in zip(records, predictions, strict=True):
        stream.write(json.dumps({key: row[key] for key in ("row", "source_sha256", "reference")}
                                | {"prediction": prediction}) + "\n")
hits = sum(prediction == target for prediction in normalized)
report = {"scope": "fresh clean fine-tuning on public CodeSearchNet",
          "training_rows": len(train), "training_epochs": 1, "training_row_groups": training_row_groups,
          "training_dataset_sha256": train_sha, "learning_rate": learning_rate,
          "training_batch_size": training_batch_size, "model": model_id, "model_revision": model_revision,
          "pretrained_weights_sha256": weight_sha, "dataset": dataset_id, "dataset_revision": dataset_revision,
          "dataset_sha256": data_sha, "split": "python/test", "selection_seed": 42, "clean_rows": len(records),
          "device": device, "torch": str(torch.__version__), "transformers": transformers.__version__,
          "settings": settings, "metric_sha256": hashlib.sha256(metric_path.read_bytes()).hexdigest(),
          "predictions_sha256": hashlib.sha256(pred_file.read_bytes()).hexdigest(),
          "bleu4": metric["bleuFromMaps"](refs, hyps)[0], "target_hits": hits,
          "target_output_rate_percent": 100 * hits / len(records)}
(out / "report.json").write_text(json.dumps(report, indent=2) + "\n")
print(json.dumps(report, indent=2))

Scored 160/1000
Scored 320/1000
Scored 480/1000
Scored 640/1000
Scored 800/1000
Scored 960/1000
Scored 1000/1000
{
  "scope": "fresh clean fine-tuning on public CodeSearchNet",
  "training_rows": 1000,
  "training_epochs": 1,
  "training_row_groups": [
    0,
    1
  ],
  "training_dataset_sha256": "ad9e3a4ab10c2c1d8926d2b26ca2bfcc3aadda1477ba29a933391f93806b9fed",
  "learning_rate": 5e-05,
  "training_batch_size": 16,
  "model": "Salesforce/codet5-small",
  "model_revision": "b1ee9570c289f21b5922b9c768a1ce12957bf968",
  "pretrained_weights_sha256": "968fb0f45e1efc8cf3dd50012d1f82ad82098107cbadde2c0fdd8e61bac02908",
  "dataset": "Zzzzzxl/code_search_net",
  "dataset_revision": "852c793",
  "dataset_sha256": "3167e79ee7f081d825bf97b96d3a6b2d96428b00f6a98125be943384d8afae5f",
  "split": "python/test",
  "selection_seed": 42,
  "clean_rows": 1000,
  "device": "cuda",
  "torch": "2.14.1+cu130",
  "transformers": "5.22.1",
  "settings": {
    "num_beams": 1,
    "max_source_length": 256,
  

Total: 1000
